In [1]:
# ============================================================
#  IMPORTS + BASIC UTILITIES
# ============================================================
import numpy as np

# Wrap utility for angular operators (kept for compatibility)
def wrap(theta):
    return (theta + np.pi) % (2*np.pi) - np.pi

rng = np.random.default_rng(12345)


In [2]:
# ============================================================
#  PATHOLOGICAL NON-LOCAL OPERATOR
# ============================================================

def make_pathological_operator(J=5, noise=0.05):
    """
    J = number of distant points to couple to
    noise = magnitude of random sign flips + discontinuities
    """
    def P(x, X):
        """
        x : current point (scalar or vector)
        X : full trajectory array (global state)
        """
        N = len(X)

        # choose J distant indices
        idx = rng.choice(N, size=J, replace=False)

        # asymmetric, sign-flipping weights
        w = rng.uniform(-1, 1, size=J) * rng.choice([-1, 1], size=J)

        # non-local update
        update = np.sum(w * (X[idx] - x))

        # inject discontinuity
        jump = rng.uniform(-noise, noise)

        return x + update + jump

    return P

P_path = make_pathological_operator()


In [3]:
# ============================================================
#  SYNTHETIC TRAJECTORY FOR AC-CR INVARIANT TESTING
# ============================================================

def generate_curve(N=500):
    """
    Simple 2D curve with curvature variation.
    """
    t = np.linspace(0, 4*np.pi, N)
    x = np.stack([
        np.cos(t) + 0.2*np.sin(3*t),
        np.sin(t) + 0.2*np.cos(5*t)
    ], axis=1)
    return x

X0 = generate_curve()


In [4]:
# ============================================================
#  AC-CR INVARIANTS ON SYNTHETIC CURVE
#  v(t), a(t), kappa(t), AC_CR(t), T(t), chi(t), J(t), T_c
# ============================================================

def compute_derivatives(X, dt):
    """
    X : (N, d) array of positions
    dt: time step
    Returns v(t), a(t)
    """
    v = np.gradient(X, dt, axis=0)
    a = np.gradient(v, dt, axis=0)
    return v, a

def curvature_2d(v, a):
    """
    2D curvature using cross product magnitude.
    kappa(t) = ||v x a|| / ||v||^3
    """
    vx, vy = v[:, 0], v[:, 1]
    ax, ay = a[:, 0], a[:, 1]
    cross = vx * ay - vy * ax
    v_norm = np.linalg.norm(v, axis=1)
    # avoid division by zero
    eps = 1e-8
    kappa = np.abs(cross) / np.maximum(v_norm**3, eps)
    return kappa

def AC_CR_signal(kappa, a):
    """
    AC-CR(t) = kappa(t) / ||a(t)||
    """
    a_norm = np.linalg.norm(a, axis=1)
    eps = 1e-8
    return kappa / np.maximum(a_norm, eps)

def thinning_T(v, a):
    """
    T(t) via directional radii in 2D.
    Here we approximate normal directions using
    a single normal vector per point.
    """
    v_norm = np.linalg.norm(v, axis=1)
    eps = 1e-8
    # unit tangent
    t_hat = v / np.maximum(v_norm[:, None], eps)
    # normal via rotation by +90 degrees
    n_hat = np.stack([-t_hat[:, 1], t_hat[:, 0]], axis=1)
    a_dot_n = np.abs(np.sum(a * n_hat, axis=1))
    R = v_norm**2 / np.maximum(a_dot_n, eps)
    # in 2D we have one normal; treat T as that radius
    return R

def normalize_signal(s):
    s_min = np.min(s)
    s_max = np.max(s)
    eps = 1e-8
    return (s - s_min) / np.maximum(s_max - s_min, eps)

def joint_signal(AC_CR, T):
    AC_n = normalize_signal(AC_CR)
    T_n  = normalize_signal(T)
    return AC_n * T_n

def transition_time(AC_CR, T, t, theta1, theta2):
    """
    T_c = inf { t >= t0 : AC_CR(t) < theta1 and T(t) < theta2 }
    """
    mask = (AC_CR < theta1) & (T < theta2)
    if np.any(mask):
        idx = np.argmax(mask)  # first True
        return t[idx]
    else:
        return None

# --- Compute invariants on X0 ---
N = X0.shape[0]
t = np.linspace(0, 1.0, N)
dt = t[1] - t[0]

v0, a0      = compute_derivatives(X0, dt)
kappa0      = curvature_2d(v0, a0)
AC_CR0      = AC_CR_signal(kappa0, a0)
T0          = thinning_T(v0, a0)
chi0        = T0 / np.maximum(AC_CR0, 1e-8)
J0          = joint_signal(AC_CR0, T0)
Tc0         = transition_time(AC_CR0, T0, t, theta1=0.2, theta2=0.2)

Tc0


np.float64(0.01002004008016032)

In [6]:
# ============================================================
#  APPLY PATHOLOGICAL OPERATOR STACK TO X0  (VECTOR-SAFE)
# ============================================================

def apply_pathological_stack(X, P, steps=10):
    """
    X     : (N, d) trajectory array
    P     : pathological operator P(x, X)
    steps : number of operator applications
    """
    X_new = X.copy()
    N = len(X_new)

    for _ in range(steps):
        X_tmp = X_new.copy()
        for i in range(N):
            X_tmp[i] = P(X_new[i], X_new)
        X_new = X_tmp

    return X_new

# --- FIXED PATHOLOGICAL OPERATOR (VECTOR-COMPATIBLE) ---
def make_pathological_operator(J=5, noise=0.05):
    def P(x, X):
        N = len(X)

        # choose J distant indices
        idx = rng.choice(N, size=J, replace=False)

        # asymmetric, sign-flipping weights, reshaped to (J,1)
        w = (rng.uniform(-1, 1, size=J) * rng.choice([-1, 1], size=J))[:, None]

        # non-local vector update
        update = np.sum(w * (X[idx] - x), axis=0)

        # inject discontinuity
        jump = rng.uniform(-noise, noise, size=x.shape)

        return x + update + jump

    return P

P_path = make_pathological_operator()

# --- Apply 10 rounds of the pathological operator ---
X_attacked = apply_pathological_stack(X0, P_path, steps=10)

X_attacked[:5]   # show first few points


array([[  327.85460699,  -112.36597481],
       [ 1760.31465916, -2049.86994479],
       [  607.33580285,   242.8293748 ],
       [ -870.24869939,  -412.6520955 ],
       [   22.72111082,  -333.45465343]])

In [7]:
# ============================================================
#  AC-CR INVARIANTS ON ATTACKED TRAJECTORY X_attacked
# ============================================================

# reuse t, dt from earlier
vA, aA      = compute_derivatives(X_attacked, dt)
kappaA      = curvature_2d(vA, aA)
AC_CRA      = AC_CR_signal(kappaA, aA)
TA          = thinning_T(vA, aA)
chiA        = TA / np.maximum(AC_CRA, 1e-8)
JA          = joint_signal(AC_CRA, TA)
TcA         = transition_time(AC_CRA, TA, t, theta1=0.2, theta2=0.2)

TcA


In [8]:
print("Tc0 =", Tc0)
print("TcA =", TcA)


Tc0 = 0.01002004008016032
TcA = None


In [9]:
# ============================================================
#  COMPARE PRE-ATTACK VS POST-ATTACK INVARIANTS
# ============================================================

def summarize(name, arr):
    return f"{name}: min={np.min(arr):.4e}, max={np.max(arr):.4e}, mean={np.mean(arr):.4e}"

print("=== AC-CR INVARIANT COMPARISON ===\n")

print("Original:")
print(summarize("AC_CR0", AC_CR0))
print(summarize("T0", T0))
print(summarize("chi0", chi0))
print(summarize("J0", J0))
print(f"Tc0 = {Tc0}")

print("\nAttacked:")
print(summarize("AC_CRA", AC_CRA))
print(summarize("TA", TA))
print(summarize("chiA", chiA))
print(summarize("JA", JA))
print(f"TcA = {TcA}")


=== AC-CR INVARIANT COMPARISON ===

Original:
AC_CR0: min=6.1213e-06, max=2.7424e+00, mean=4.8757e-02
T0: min=9.3775e-04, max=4.4098e+02, mean=4.8332e+00
chi0: min=3.4194e-04, max=7.2040e+07, mean=2.3463e+05
J0: min=0.0000e+00, max=2.9274e-05, mean=2.0260e-06
Tc0 = 0.01002004008016032

Attacked:
AC_CRA: min=1.4089e-14, max=2.9236e-09, mean=2.8070e-11
TA: min=1.4423e+00, max=3.0910e+06, mean=1.1552e+04
chiA: min=1.4423e+08, max=3.0910e+14, mean=1.1552e+12
JA: min=0.0000e+00, max=9.7437e-06, mean=3.2052e-07
TcA = None
